# 04 — Evaluation

**Exploration 20 · Crop 112/224 · Graph radius 224 · G3 vs GAT**

Use the original **Stage 1** best checkpoints and evaluate both Exploration 20 **Stage 2** variants with the **same detection and ROI metrics** as the old notebook.

- **Stage 1:** ConvNeXt-Tiny and ConvNeXt-Base (original project).
- **Stage 2:** G3 and GAT (Exploration 20).
- **Cross-validation:** Plans A/B, folds 0–3, unchanged ROI membership.
- **Evaluation:** 32 end-to-end combinations (predicted detector centroids, not ground-truth centroids).
- **Output:** `results/evaluation/history.csv` inside Exploration 20; the original project is not overwritten.

## 1. Environment and paths

### 1.1 Mount Drive

In [1]:
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive


drive.mount("/content/drive")


Mounted at /content/drive


### 1.2 Project locations and dependencies

In [2]:
# Original Stage 1 repository and checkpoint locations — unchanged.
original_project_dir = Path(
    "/content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline"
)

# Exploration 20 holds the updated Stage 2 code and checkpoints.
project_dir = Path(
    "/content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT"
)

assert original_project_dir.is_dir(), f"Missing original project: {original_project_dir}"
assert project_dir.is_dir(), f"Missing Exploration 20 project: {project_dir}"
assert (original_project_dir / "stage1").is_dir(), "Original Stage 1 code is required."
assert (project_dir / "stage2").is_dir(), "Exploration 20 Stage 2 code is required."

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', '/content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/requirements.txt'], returncode=0)

### 1.3 Import Stage 1 from the original project and Stage 2 from Exploration 20

In [3]:
import os

os.environ["PUMA_PROJECT_DIR"] = str(project_dir)
os.environ["PUMA_RESULTS_DIR"] = str(project_dir / "results")

for root in (str(original_project_dir), str(project_dir)):
    if root in sys.path:
        sys.path.remove(root)

sys.path.insert(0, str(original_project_dir))
sys.path.insert(0, str(project_dir))

import stage1
import stage2
import common

assert Path(stage1.__file__).resolve().parent == original_project_dir / "stage1", (
    "Restart the runtime: wrong Stage 1 module loaded."
)
assert Path(stage2.__file__).resolve().parent == project_dir / "stage2", (
    "Restart the runtime: wrong Stage 2 module loaded."
)
assert Path(common.__file__).resolve().parent == project_dir / "common", (
    "Restart the runtime: wrong common module loaded."
)


### 1.4 Resolve checkpoint, cache, and output paths

In [4]:
from common.settings import paths as get_paths

paths = get_paths()

original_results = original_project_dir / "results"
stage1_cache = original_project_dir / "preprocess" / "stage1"
probability_dir = stage1_cache.parent / "nito_maps"
images_dir = paths["images"]
exploration_results = paths["results"]

assert torch.cuda.is_available(), "Select a Colab GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 1 checkpoint root:", original_results / "stage1")
print("Stage 1 cache:", stage1_cache)
print("Stage 2 checkpoint root:", exploration_results / "stage2")
print("NiTo maps:", probability_dir)
print("Evaluation output:", exploration_results / "evaluation" / "history.csv")


GPU: NVIDIA L4
Stage 1 checkpoint root: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/results/stage1
Stage 1 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage1
Stage 2 checkpoint root: /content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/results/stage2
NiTo maps: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/nito_maps
Evaluation output: /content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/results/evaluation/history.csv


## 2. Evaluation configuration

### 2.1 Imports

In [5]:
import csv
import gc
import json

import numpy as np

from common.resources import gpu_batch_size
from common.run_history import SCHEMA, file_fingerprint

from stage1.config import DETECTOR_REVISION
from stage1.dataset import Cache
from stage1.decode import decode_stage1_candidates
from stage1.evaluate import collect_candidates, choose_tile_batch_size
from stage1.models.build import build_model
from stage1.preprocess import _index_files
from stage1.splits import load_splits as load_stage1_splits
from stage1.target_core import _read_rgb

from stage2.data import class_names
from stage2.features import Extractor
from stage2.graph import build as build_graph, gpu_bundle
from stage2.metrics import roi_scores
from stage2.model import NucleiFusion
from stage2.nito import Nito
from stage2.settings import (
    FEATURE_REVISION,
    FOVS,
    MODEL_REVISIONS,
    VARIANTS,
    graph_settings,
)
from stage2.splits import load_splits as load_stage2_splits


### 2.2 Same plans, folds and metrics as the original evaluation

In [6]:
architectures = ("convnext_tiny", "convnext_base")
plans = ("A", "B")
variants = VARIANTS  # G3 and GAT
folds = range(4)

METRICS = (
    "accuracy",
    "precision",
    "recall",
    "f1",
    "macro_f1_nuclei",
    "roi_f1",
)

COLUMNS = [
    "variant",
    "plan",
    "architecture",
    "fold",
    "num_rois",
    "tp",
    "fp",
    "fn",
    *METRICS,
    *[
        f"{label}_{metric}"
        for label in class_names
        for metric in ("accuracy", "precision", "recall", "f1")
    ],
    *[f"{label}_official_roi_f1" for label in class_names],
]


### 2.3 Checkpoint locations and metric formatting

In [7]:
def stage1_checkpoint_path(plan, arch, fold):
    """Exact Stage 1 checkpoint path from the original evaluation."""
    return (
        original_results
        / "stage1"
        / f"plan_{plan}"
        / arch
        / f"fold_{fold}"
        / "best.pt"
    )


def stage2_checkpoint_path(mode, plan, fold):
    """Exploration 20 best checkpoint for each Stage 2 graph variant."""
    return (
        exploration_results
        / "stage2"
        / f"r224_{mode.lower()}"
        / f"plan_{plan}"
        / f"fold_{fold}"
        / "best.pt"
    )


In [8]:
def row_metrics(metrics):
    """Preserve the original CSV metric definitions and per-class columns."""
    row = {key: metrics[key] for key in ("tp", "fp", "fn", *METRICS)}

    for name in class_names:
        values = metrics["per_class"][name]
        row[f"{name}_accuracy"] = values["tp"] / max(
            1, values["tp"] + values["fp"] + values["fn"]
        )
        row.update({
            f"{name}_{metric}": values[metric]
            for metric in ("precision", "recall", "f1")
        })
        row[f"{name}_official_roi_f1"] = metrics["f1_nuclei"]["nuclei_" + name]

    return row


## 3. Validate input data and checkpoints

### 3.1 Verify original image-source alignment

In [9]:
def check_image_source_alignment(cache, images):
    """Same raw-source validation as old stage2.preprocess.verify_images_against_stage1."""
    expected = {
        row[0]: row[4]
        for row in cache.meta.get("source_file_metadata", [])
        if len(row) >= 5 and Path(row[1]).suffix.lower() in (".tif", ".tiff")
    }
    digests = {}
    for index, name in enumerate(cache.meta["names"]):
        if name not in images:
            raise FileNotFoundError(f"Missing Stage 1 raw TIFF: {name}")
        path = Path(images[name])
        digest = file_fingerprint(path)
        matches = (digest == expected[name]) if name in expected else np.array_equal(
            _read_rgb(path, size=cache.size), cache.images[index]
        )
        if not matches:
            raise ValueError(f"Stage 1 raw image/cache mismatch: {name}")
        digests[name] = digest
    return digests


### 3.2 Validate cache provenance, Stage 1 checkpoints and both Stage 2 variants

In [10]:
def validate_inputs():
    cache = Cache(stage1_cache)
    if cache.meta.get("target_version") != "puma_one_feature_one_nucleus_geometry_centroid_v2":
        raise ValueError("Original Stage 1 cache has an outdated target version")

    manifest_path = paths["stage2_cache"] / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    if (
        manifest.get("feature_revision") != FEATURE_REVISION
        or tuple(manifest.get("fovs", ())) != FOVS
        or manifest.get("graph_settings") != graph_settings()
        or tuple(manifest.get("class_order", ())) != class_names
        or set(manifest.get("names", ())) != set(cache.meta["names"])
    ):
        raise ValueError("Exploration 20 preprocessing manifest does not match E20 code")
    if manifest.get("graph_sha256") != file_fingerprint(paths["stage2_cache"] / "graph.npz"):
        raise ValueError("Exploration 20 graph cache checksum mismatch")

    missing = [
        str(p) for plan in plans for fold in range(4)
        for p in (
            *[stage1_checkpoint_path(plan, arch, fold) for arch in architectures],
            *[stage2_checkpoint_path(mode, plan, fold) for mode in variants],
        ) if not p.is_file()
    ]
    if missing:
        raise FileNotFoundError(f"Missing {len(missing)} best.pt checkpoints: {missing[:6]}")

    images = _index_files(Path(images_dir), {".tif", ".tiff"})
    selected_images = {name: images[name] for name in cache.meta["names"]}
    image_sources = check_image_source_alignment(cache, selected_images)
    encoder_sources = {
        "uni2": file_fingerprint(Path(paths["uni2"])),
        "nito": {
            f"{group}{i}": file_fingerprint(Path(paths["nito"]) / f"{group}{i}" / "checkpoint_epoch1.pth")
            for group in ("classifier", "Primary", "Metastatic")
            for i in range(5)
        },
    }
    if manifest.get("image_sources") != image_sources or manifest.get("encoder_sources") != encoder_sources:
        raise ValueError("Exploration 20 cache uses different images or encoder weights")

    for plan in plans:
        old_splits = load_stage1_splits(plan)
        new_splits = load_stage2_splits(plan)
        for fold in range(4):
            for key in ("train", "val"):
                if set(old_splits[fold][key]) != set(new_splits[fold][key]):
                    raise ValueError(f"Stage 1 and Exploration 20 {plan}/fold{fold} {key} ROI split mismatch")

            for arch in architectures:
                ckpt_path = stage1_checkpoint_path(plan, arch, fold)
                ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
                cfg = ckpt.get("config", {})
                if (
                    ckpt.get("schema") != SCHEMA
                    or cfg.get("cache_id") != cache.meta["id"]
                    or cfg.get("detector_revision") != DETECTOR_REVISION
                    or cfg.get("arch") != arch
                    or cfg.get("plan") != plan
                    or cfg.get("fold") != fold
                    or set(cfg.get("train", ())) != set(old_splits[fold]["train"])
                    or set(cfg.get("val", ())) != set(old_splits[fold]["val"])
                    or ckpt.get("operating_point", {}).get("selection_set") != "train"
                    or "threshold" not in ckpt.get("operating_point", {})
                    or "suppression_radius" not in ckpt.get("operating_point", {})
                ):
                    raise ValueError(f"Incompatible original Stage 1 checkpoint: {ckpt_path}")
                del ckpt

            for mode in variants:
                ckpt_path = stage2_checkpoint_path(mode, plan, fold)
                ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
                if (
                    ckpt.get("schema") != SCHEMA
                    or ckpt.get("graph_mode") != mode
                    or ckpt.get("model_revision") != MODEL_REVISIONS[mode]
                    or ckpt.get("feature_revision") != FEATURE_REVISION
                    or ckpt.get("graph_settings") != graph_settings()
                    or ckpt.get("feature_source_id") != manifest.get("source_id")
                    or tuple(ckpt.get("class_names", ())) != class_names
                    or ckpt.get("plan") != plan
                    or ckpt.get("fold") != fold
                    or set(ckpt.get("train_rois", ())) != set(new_splits[fold]["train"])
                    or set(ckpt.get("val_rois", ())) != set(new_splits[fold]["val"])
                    or ckpt.get("image_sources") != image_sources
                    or ckpt.get("encoder_sources") != encoder_sources
                ):
                    raise ValueError(f"Incompatible Exploration 20 {mode} checkpoint: {ckpt_path}")
                del ckpt

    return cache, images


### 3.3 Run validation before inference

In [11]:
cache, images = validate_inputs()

print("Validation complete: original Stage 1 + Exploration 20 G3/GAT.")
print("Plans A/B, four folds, source images and checkpoints verified.")


Validation complete: original Stage 1 + Exploration 20 G3/GAT.
Plans A/B, four folds, source images and checkpoints verified.


## 4. Stage 2 classifier inference

Use the Exploration 20 model and the training-fold Tier-A normalisation. Graph radius, K and sigma come from Exploration 20’s saved settings, so G3 and GAT share the same neighborhood geometry.

In [12]:
class Exploration20Classifier:
    """Mode-aware counterpart of the original Stage 2 inference.Classifier."""

    def __init__(self, checkpoint, mode, device):
        self.device = torch.device(device)
        self.mode = mode
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        self.mean = np.asarray(state["tier_mean"], dtype=np.float32)
        self.std = np.asarray(state["tier_std"], dtype=np.float32)
        if (
            self.mean.shape != (16,) or self.std.shape != (16,)
            or not np.isfinite(self.mean).all() or not np.isfinite(self.std).all()
            or np.any(self.std <= 0)
        ):
            raise ValueError(f"Invalid training-only Tier-A normalization in {checkpoint}")
        if state.get("graph_mode") != mode or state.get("model_revision") != MODEL_REVISIONS[mode]:
            raise ValueError(f"Wrong Stage 2 mode/revision: {checkpoint}")
        self.model = NucleiFusion(mode=mode).to(self.device).eval()
        self.model.load_state_dict(state["model"], strict=True)
        self.batch_size = 0

    @torch.inference_mode()
    def predict(self, features, batch_size=0):
        if batch_size < 0:
            raise ValueError("batch_size must be >= 0")
        xy = np.asarray(features["xy"], dtype=np.float32).reshape(-1, 2)
        if len(xy) == 0:
            return np.empty((0, len(class_names)), dtype=np.float32)
        tier = ((features["tier_raw"] - self.mean) / self.std).astype(np.float32)
        settings = graph_settings()
        # Exactly the inference-time neighborhood logic from the original classifier.
        graph = build_graph(
            xy, np.zeros(len(xy), dtype=np.int32),
            radius=settings["radius"], k=settings["k"], sigma=settings["sigma"],
        )
        data = gpu_bundle({**features, "tier": tier}, graph, self.device, mode=self.mode)
        try:
            if batch_size == 0:
                if not self.batch_size or len(xy) > self.batch_size:
                    def run(size):
                        ids = torch.arange(size, device=self.device) % len(xy)
                        logits = self.model(ids, data)
                        del ids, logits

                    self.batch_size = gpu_batch_size(
                        self.device, run, limit=min(2048, len(xy))
                    )
                    print(f"E20 {self.mode} inference batch size: {self.batch_size}", flush=True)
                batch_size = self.batch_size

            logits = []
            for left in range(0, len(xy), batch_size):
                ids = torch.arange(left, min(left + batch_size, len(xy)), device=self.device)
                logits.append(self.model(ids, data).float().cpu())
            return torch.cat(logits).softmax(1).numpy()
        finally:
            del data


### 4.1 Prepare arrays for the original ROI scoring function

In [13]:
def _prepare_metric_arrays(values):
    return {
        key: np.asarray(rows).reshape(-1, 2) if key == "xy" else np.asarray(rows)
        for key, rows in values.items()
    }


## 5. Stage 1 detector — unchanged

### 5.1 Detect nucleus candidates once per fold

The original detector model, calibrated proposal threshold, NMS suppression radius and maximum candidate count are reused without modification.

In [14]:
def detect_stage1_proposals(cache, plan, arch, fold, device):
    """Run the original Stage 1 detection pipeline on validation ROIs."""
    det_path = stage1_checkpoint_path(plan, arch, fold)
    det = torch.load(det_path, map_location="cpu", weights_only=False)
    cfg = det["config"]
    setting = det["operating_point"]
    model = build_model(arch, pretrained=False).to(device).eval()
    model.load_state_dict(det["model"], strict=True)
    model.pretrained = bool(cfg["pretrained"])
    tile = int(cfg["tile_size"])
    overlap = int(cfg["overlap"])
    amp = bool(cfg.get("amp", True))
    batch = choose_tile_batch_size(model, device, tile, amp=amp)
    names = load_stage1_splits(plan)[fold]["val"]

    proposals = {}
    for i, name in enumerate(names, 1):
        candidates = collect_candidates(
            model, cache, name, tile=tile, overlap=overlap,
            device=device, amp=amp, batch_size=batch,
            minimum_proposal_score=min(cfg["train_thresholds"]),
        )
        predicted = decode_stage1_candidates(
            candidates, threshold=setting["threshold"],
            suppression_radius_pixels=setting["suppression_radius"],
            max_candidates=4000,
        )
        xy = (
            np.column_stack((predicted["x"], predicted["y"])).astype(np.float32)
            if len(predicted) else np.empty((0, 2), dtype=np.float32)
        )
        proposals[name] = (xy, predicted["heatmap_score"].copy())
        if i % 10 == 0:
            print(f"Detect {plan}/{arch}/fold{fold}: {i}/{len(names)}", flush=True)
    del model, det
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return names, proposals


## 6. Stage 2 classification — G3 and GAT

### 6.1 Extract UNI2/NiTo features once and classify with both checkpoints

Both variants receive identical detector proposals. The image feature extractor is shared, but each variant uses its own trained parameters.

In [15]:
def classify_stage2_proposals(cache, images, plan, arch, fold, device, names, proposals):
    """Evaluate both Exploration 20 Stage 2 variants on the same proposals."""
    extractor = Extractor(paths["uni2"], paths["nito"], device=device,
                          probability_dir=probability_dir)
    classifiers = {
        mode: Exploration20Classifier(stage2_checkpoint_path(mode, plan, fold), mode, device)
        for mode in variants
    }
    gt = {"roi": [], "xy": [], "label": []}
    predictions = {
        mode: {"roi": [], "xy": [], "label": [], "score": []}
        for mode in variants
    }

    try:
        for i, name in enumerate(names, 1):
            xy, scores = proposals[name]
            features = extractor.extract(images[name], xy)
            truth = cache.gt_for(name)
            mapped = truth["class_id"].copy()
            # Same Stage 1 -> Stage 2 epithelium/endothelium remapping.
            mapped[mapped == 7] = 11
            mapped[mapped == 8] = 7
            mapped[mapped == 11] = 8
            gt["roi"].extend([name] * len(truth))
            gt["xy"].extend(np.column_stack((truth["x"], truth["y"])))
            gt["label"].extend(mapped)

            for mode, classifier in classifiers.items():
                proba = classifier.predict(features)
                labels = proba.argmax(1) if len(proba) else np.empty(0, dtype=int)
                confidence = (
                    (proba.max(1) * scores).clip(0, 1)
                    if len(proba) else np.empty(0, dtype=float)
                )
                pred = predictions[mode]
                pred["roi"].extend([name] * len(labels))
                pred["xy"].extend(xy)
                pred["label"].extend(labels)
                pred["score"].extend(confidence)

            if i % 10 == 0:
                print(f"Classify G3/GAT {plan}/{arch}/fold{fold}: {i}/{len(names)}", flush=True)
    finally:
        extractor.close()
        del extractor, classifiers
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    gt = _prepare_metric_arrays(gt)
    return {
        mode: roi_scores(gt, _prepare_metric_arrays(predictions[mode]), roi_names=names)
        for mode in variants
    }


### 6.2 Evaluate one Plan / Stage 1 architecture / fold

In [16]:
def evaluate_one_fold(cache, images, plan, arch, fold, device):
    names, proposals = detect_stage1_proposals(
        cache, plan, arch, fold, device
    )

    return classify_stage2_proposals(
        cache, images, plan, arch, fold, device, names, proposals
    )


## 7. Out-of-fold (OOF) aggregation

### 7.1 Same per-class ROI F1 and mean/std aggregation as the original evaluator

In [17]:
def _summarize(rows, reports, mode, plan, arch):
    """Original fast_evaluation OOF + fold mean/std, grouped by E20 graph variant."""
    items = [reports[(mode, plan, arch, fold)] for fold in range(4)]
    detail = {}
    for item in items:
        for roi, classes in item["per_roi_class"].items():
            if roi in detail:
                raise ValueError(f"Duplicate OOF validation ROI: {roi}")
            detail[roi] = classes
    expected = set().union(*(set(x["val"]) for x in load_stage1_splits(plan).values()))
    if set(detail) != expected:
        raise ValueError(f"OOF ROI coverage mismatch for {mode}/{plan}/{arch}")
    f1s = {
        name: float(np.mean([classes[name]["f1"] for classes in detail.values()]))
        for name in class_names
    }
    observed = set().union(*(
        set(item.get("observed_nuclei_classes", class_names)) for item in items
    ))
    macro = float(np.mean([f1s[name] for name in observed])) if observed else 0.0
    # In the original evaluator, macro_f1_nuclei is the official per-ROI class macro.
    result = {
        "variant": mode, "plan": plan, "architecture": arch, "fold": "OOF",
        "num_rois": len(detail), "macro_f1_nuclei": macro,
        "roi_f1": float(np.mean([item["per_roi"][roi]["f1"]
                                for item in items for roi in item["per_roi"]])),
    }
    result.update({f"{name}_official_roi_f1": score for name, score in f1s.items()})
    rows.append(result)
    for kind, agg in (("mean", lambda x: float(np.mean(x))),
                      ("std", lambda x: float(np.std(x, ddof=1)))):
        row = {
            "variant": mode, "plan": plan, "architecture": arch,
            "fold": kind, "num_rois": sum(x["num_rois"] for x in items),
        }
        for metric in METRICS:
            row[metric] = agg([item[metric] for item in items])
        rows.append(row)
    print(
        f"OOF {mode}/{plan}/{arch}: "
        f"macro F1 {macro:.6f}, ROI F1 {result['roi_f1']:.6f}, "
        f"ROIs={len(detail)}",
        flush=True,
    )


### 7.2 Run all combinations and write the CSV

Each detector is evaluated once per fold, then both G3 and GAT predictions are scored. Original Stage 1 checkpoints and results are not modified.

In [18]:
def evaluate_all():
    Nito(paths["nito"], device="cuda").prepare_maps(
        {name: images[name] for name in cache.meta["names"]}, probability_dir
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows, reports = [], {}
    for plan in plans:
        for fold in folds:
            for arch in architectures:
                print(f"EVAL {plan}/{arch}/fold{fold} | E20 G3 + GAT", flush=True)
                result = evaluate_one_fold(cache, images, plan, arch, fold, device)
                for mode in variants:
                    report = result[mode]
                    reports[(mode, plan, arch, fold)] = report
                    rows.append({
                        "variant": mode, "plan": plan, "architecture": arch, "fold": fold,
                        "num_rois": report["num_rois"], **row_metrics(report),
                    })
    for mode in variants:
        for plan in plans:
            for arch in architectures:
                _summarize(rows, reports, mode, plan, arch)
    output = exploration_results / "evaluation" / "history.csv"
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    print(f"Evaluation saved ONE file: {output}", flush=True)
    return rows


## 8. Run the complete evaluation

**32 combinations:** 2 Stage 1 architectures × 2 Stage 2 variants × 2 plans × 4 folds.

In [19]:
rows = evaluate_all()

print("Saved:", exploration_results / "evaluation" / "history.csv")


Reusing verified NiTo maps
EVAL A/convnext_tiny/fold0 | E20 G3 + GAT
Detect A/convnext_tiny/fold0: 10/52
Detect A/convnext_tiny/fold0: 20/52
Detect A/convnext_tiny/fold0: 30/52
Detect A/convnext_tiny/fold0: 40/52
Detect A/convnext_tiny/fold0: 50/52
UNI2 batch size: 256 | device: cuda
E20 G3 inference batch size: 256
E20 GAT inference batch size: 256
E20 G3 inference batch size: 512
E20 GAT inference batch size: 512
E20 G3 inference batch size: 512
E20 GAT inference batch size: 512
E20 G3 inference batch size: 512
E20 GAT inference batch size: 512
E20 G3 inference batch size: 512
E20 GAT inference batch size: 512
E20 G3 inference batch size: 512
E20 GAT inference batch size: 512
Classify G3/GAT A/convnext_tiny/fold0: 10/52
E20 G3 inference batch size: 1024
E20 GAT inference batch size: 1024
Classify G3/GAT A/convnext_tiny/fold0: 20/52
Classify G3/GAT A/convnext_tiny/fold0: 30/52
Classify G3/GAT A/convnext_tiny/fold0: 40/52
Classify G3/GAT A/convnext_tiny/fold0: 50/52
EVAL A/convnext_bas

## 9. Review the results

### 9.1 OOF scores by plan, detector architecture and Stage 2 variant

`macro_f1_nuclei` and `roi_f1` are calculated with the same class-aware ROI scoring as the original evaluator, using *detector-predicted* coordinates.

In [20]:
import pandas as pd
from IPython.display import display

summary = pd.DataFrame([row for row in rows if row["fold"] == "OOF"])
summary = summary[
    ["variant", "plan", "architecture", "num_rois", "macro_f1_nuclei", "roi_f1"]
]

print("Exploration 20 — end-to-end out-of-fold results")
display(summary.sort_values(["plan", "architecture", "variant"]).reset_index(drop=True))


Exploration 20 — end-to-end out-of-fold results


,variant,plan,architecture,num_rois,macro_f1_nuclei,roi_f1
0,G3,A,convnext_base,205,0.287014,0.805692
1,GAT,A,convnext_base,205,0.285641,0.802288
2,G3,A,convnext_tiny,205,0.285581,0.803856
3,GAT,A,convnext_tiny,205,0.283621,0.800383
4,G3,B,convnext_base,206,0.285232,0.801668
5,GAT,B,convnext_base,206,0.282220,0.796627
6,G3,B,convnext_tiny,206,0.285345,0.801719
7,GAT,B,convnext_tiny,206,0.281391,0.796456


### 9.2 Overall comparison — which variant has the higher ROI F1?

Average each variant over the same 2 plans and 2 Stage 1 architectures, giving each combination equal weight.

In [21]:
ranked = summary.groupby("variant", as_index=False)[
    ["roi_f1", "macro_f1_nuclei"]
].mean()
ranked = ranked.sort_values("roi_f1", ascending=False).reset_index(drop=True)

print("Mean OOF scores: equal weighting across Plan A/B and Tiny/Base")
display(ranked)

if np.isclose(ranked.loc[0, "roi_f1"], rantdrked.loc[1, "roi_f1"], rtol=0, atol=1e-12):
    print("WINNER by end-to-end ROI F1: TIE")
else:
    print("WINNER by end-to-end ROI F1:", ranked.loc[0, "variant"])

print("Per-fold and per-class metrics:", exploration_results / "evaluation" / "history.csv")


Mean OOF scores: equal weighting across Plan A/B and Tiny/Base


,variant,roi_f1,macro_f1_nuclei
0,G3,0.803234,0.285793
1,GAT,0.798939,0.283218


WINNER by end-to-end ROI F1: G3
Per-fold and per-class metrics: /content/drive/MyDrive/Research/PUMA/Exploration_20_Crop112-224_Graph224_G3_vs_GAT/results/evaluation/history.csv


## 10. Release the Colab runtime (optional)

Run this cell **only after** you have checked the results above.

In [22]:
from google.colab import runtime

runtime.unassign()
